In [ ]:
# --- config ---
INPUT_GPKG  = "nuclide2.0.gpkg"      # your input
INPUT_LAYER = "nuclidecsv"           # layer name inside the gpkg

OUT_GPKG    = "nuclide_pies.gpkg"    # output gpkg with polygon pies
OUT_LAYER   = "nuclide_pies"         # output layer name
OUT_QML     = "nuclide_pies.qml"     # style file (categorized by Th/U/K)

# pie radius scaling (in meters, since we’ll work in EPSG:3857)
MIN_R = 150.0
MAX_R = 450.0

# colors for categories
COLORS = {"Th": "#1f77b4", "U": "#2ca02c", "K": "#ff7f0e"}

# --- code ---
import math, os
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Polygon, mapping
import fiona
from fiona.crs import from_epsg

def radius_from_total(total, tmin, tmax, rmin, rmax):
    if not np.isfinite(total):
        return rmin
    if tmax <= tmin:
        return 0.5*(rmin + rmax)
    return rmin + (rmax - rmin) * ((total - tmin) / (tmax - tmin))

def sector_polygon(cx, cy, r, ang0, ang1, steps=48):
    # Build a sector polygon centered at (cx,cy), radius r, angles in radians
    if ang1 < ang0:
        ang1 += 2*math.pi
    angles = np.linspace(ang0, ang1, steps)
    coords = [(cx, cy)] + [(cx + r*math.cos(a), cy + r*math.sin(a)) for a in angles] + [(cx, cy)]
    return Polygon(coords)

# 1) Load input, ensure projected CRS (EPSG:3857) for meter-based radii
gdf = gpd.read_file(INPUT_GPKG, layer=INPUT_LAYER)
if gdf.crs is None or gdf.crs.to_epsg() != 3857:
    gdf = gdf.to_crs(3857)

# 2) Clean/standardize fields
gdf["Th_mgkg"] = pd.to_numeric(gdf["m(Th-232) (mg/kg(sample))"], errors="coerce").clip(lower=0)
gdf["U_mgkg"]  = pd.to_numeric(gdf["m(U-238) (mg/kg(sample))"],  errors="coerce").clip(lower=0)
gdf["K_mgkg"]  = pd.to_numeric(gdf["m(K-40) (mg/kg(sample))"],   errors="coerce").clip(lower=0)
gdf["Total_mgkg"] = gdf[["Th_mgkg","U_mgkg","K_mgkg"]].sum(axis=1)

t_min = float(gdf["Total_mgkg"].min())
t_max = float(gdf["Total_mgkg"].max())

# 3) Build pie-slice polygons (one feature per slice)
records = []
for _, row in gdf.iterrows():
    x, y = row.geometry.x, row.geometry.y
    th, uu, kk = float(row["Th_mgkg"] or 0), float(row["U_mgkg"] or 0), float(row["K_mgkg"] or 0)
    total = th + uu + kk
    r = radius_from_total(total, t_min, t_max, MIN_R, MAX_R)

    parts = [("Th", th), ("U", uu), ("K", kk)]
    start = -math.pi/2  # start at top
    for name, val in parts:
        if total > 0 and val > 0:
            ang_span = (val/total) * 2*math.pi
            end = start + ang_span
            poly = sector_polygon(x, y, r, start, end, steps=48)
            records.append({
                "Sample_num": str(row.get("Sample number", "")),
                "Location": row.get("Location", ""),
                "Lithology": row.get("Lithology", ""),
                "Soil_Type": row.get("Soil Type", ""),
                "Th_mgkg": th,
                "U_mgkg": uu,
                "K_mgkg": kk,
                "Total_mgkg": total,
                "category": name,             # Th / U / K
                "value": val,                 # slice value
                "fraction": val/total,        # slice fraction
                "color_hex": COLORS[name],    # for reference
                "geometry": poly
            })
        start = start + ( (val/total)*2*math.pi if total>0 else 0 )

# 4) Write to GPKG using Fiona (robust with Shapely 2)
schema = {
    "geometry": "Polygon",
    "properties": {
        "Sample_num": "str",
        "Location": "str",
        "Lithology": "str",
        "Soil_Type": "str",
        "Th_mgkg": "float",
        "U_mgkg": "float",
        "K_mgkg": "float",
        "Total_mgkg": "float",
        "category": "str",
        "value": "float",
        "fraction": "float",
        "color_hex": "str",
    },
}

if os.path.exists(OUT_GPKG):
    os.remove(OUT_GPKG)

with fiona.open(OUT_GPKG, mode="w", driver="GPKG",
                layer=OUT_LAYER, schema=schema, crs=from_epsg(3857)) as dst:
    for rec in records:
        geom = rec.pop("geometry")
        dst.write({"geometry": mapping(geom), "properties": rec})

# 5) Emit a simple QML that colors by 'category'
qml = f"""<qgis styleCategories="Symbology" version="3.28.0">
  <renderer-v2 type="categorizedSymbol" attr="category" enableorderby="0" forceraster="0" symbollevels="0">
    <categories>
      <category symbol="0" value="Th" label="Thorium-232"/>
      <category symbol="1" value="U"  label="Uranium-238"/>
      <category symbol="2" value="K"  label="Potassium-40"/>
    </categories>
    <symbols>
      <symbol alpha="1" type="fill" name="0">
        <layer pass="0" class="SimpleFill" locked="0">
          <prop k="color" v="{COLORS['Th']}"/>
          <prop k="outline_color" v="#333333"/>
          <prop k="outline_width" v="0.2"/>
          <prop k="style" v="solid"/>
        </layer>
      </symbol>
      <symbol alpha="1" type="fill" name="1">
        <layer pass="0" class="SimpleFill" locked="0">
          <prop k="color" v="{COLORS['U']}"/>
          <prop k="outline_color" v="#333333"/>
          <prop k="outline_width" v="0.2"/>
          <prop k="style" v="solid"/>
        </layer>
      </symbol>
      <symbol alpha="1" type="fill" name="2">
        <layer pass="0" class="SimpleFill" locked="0">
          <prop k="color" v="{COLORS['K']}"/>
          <prop k="outline_color" v="#333333"/>
          <prop k="outline_width" v="0.2"/>
          <prop k="style" v="solid"/>
        </layer>
      </symbol>
    </symbols>
  </renderer-v2>
</qgis>
"""
with open(OUT_QML, "w", encoding="utf-8") as f:
    f.write(qml)

print("Wrote:", OUT_GPKG, "layer:", OUT_LAYER)
print("Style:", OUT_QML)